In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
schema = StructType([
    StructField("country", StringType(), True),
    StructField("total_population", IntegerType(), True)
])

data = [("USA", 1000), ("INDIA", 9999), ("IRAN", 456789), ("INDIA", 10000), ("USA", 1000)]
df = spark.createDataFrame(data, schema=schema)

df.write.mode("overwrite").saveAsTable("test.test.population")

In [0]:
%sql
select * from population;

country,total_population_2023,total_population_2022
CHINA,null,1400000000
USA,789456123,780000000
AFRICA,99999999,null
INDIA,331002651,1300000000
USA,500000000,450000000
USA,null,280000000
INDIA,1200000000,null
CHINA,null,1450000000


##### MAX, MIN, SUM, AVG aggregate functions

In [0]:
%sql
select max(total_population_2023) as max,
        max(round(cast(total_population_2023 as double)/100000000, 2)) as max_population,
        sum(total_population_2023) as sum,
        avg(total_population_2023) as avg
from population;

max,max_population,sum,avg
1200000000,12.0,2920458773,5.840917546E8


In [0]:
from pyspark.sql.functions import col, max, round, sum, avg

df.display()
df.select(
    max(col("total_population")).alias("max_population"),
    max(round(col("total_population").cast("double") / 1000, 2)).alias("pop"),
    sum(col("total_population")).alias("sum_population"),
    avg(col("total_population")).alias("avg_population")
).display()

country,total_population
USA,1000
INDIA,9999
IRAN,456789
INDIA,10000
USA,1000


max_population,pop,sum_population,avg_population
456789,456.79,478788,95757.6


##### COUNT function

In [0]:
# COUNT --> it is used to find the count of records
# COUNT(*) == COUNT(1) --> any number in count function produces same result as *. so nothing fancy

In [0]:
%sql
select * from population;

country,total_population_2023,total_population_2022
CHINA,null,1400000000
USA,789456123,780000000
AFRICA,99999999,null
INDIA,331002651,1300000000
USA,500000000,450000000
USA,null,280000000
INDIA,1200000000,null
CHINA,null,1450000000


In [0]:
%sql
select 
    count(*) as total_count, 
    count(total_population_2023) as total_count_2023,
    count(distinct country) as count_distinct_country
from population;

total_count,total_count_2023,count_distinct_country
8,5,4


In [0]:
from pyspark.sql.functions import col, count, when, lit, countDistinct
df.display()
df1 = df.withColumn(
    "total_population",
    when (col("country") == "USA", lit(None))
    .otherwise(col("total_population"))
)
df1.display()

df1.select(
    count("*").alias("total_count"),
    count(col("total_population")).alias("total"),
    countDistinct(col("total_population")).alias("distinct")
).display()

df1.select("country").distinct().display()

country,total_population
USA,1000
INDIA,9999
IRAN,456789
INDIA,10000
USA,1000


country,total_population
USA,null
INDIA,9999
IRAN,456789
INDIA,10000
USA,null


total_count,total,distinct
5,3,3


country
USA
INDIA
IRAN


##### GROUP BY function

In [0]:
%sql
-- INSERT INTO population
-- VALUES
-- ('USA', 500000000, 450000000),
-- ('USA', 300000000, 280000000),
-- ('INDIA', 1200000000, 1100000000),
-- ('CHINA', 1500000000, 1450000000);

num_affected_rows,num_inserted_rows
4,4


In [0]:
%sql
UPDATE population 
SET 
    total_population_2023 = 
        CASE
            WHEN total_population_2023 = 300000000 then null
            else total_population_2023
        end,
    
    total_population_2022 = 
        CASE
            WHEN total_population_2022 = '1100000000' then null
            else total_population_2022
        end;

num_affected_rows
8


In [0]:
%sql
select country, count(*) from population group by country;
select country from population where total_population_2022 > 500000000 group by country;
select country, count(*), max(total_population_2023) from population group by country;

country,count(*),max(total_population_2023)
CHINA,2,null
USA,3,789456123
AFRICA,1,99999999
INDIA,2,1200000000


In [0]:
new_data = [
    ("USA", 500000000),
    ("USA", 300000000),
    ("INDIA", 1200000000),
    ("CHINA", 1500000000)
]
new_df = spark.createDataFrame(new_data, ["country","total_population"])
df1 = df1.union(new_df)

In [0]:
from pyspark.sql.functions import count, max, when, lit, col
df1.withColumn(
    "total_population",
    when (col("total_population") == 300000000, lit(None))
    .otherwise(col("total_population"))
).display()

df1.groupBy("country") \
    .agg(
        count("*"),
        max("total_population").alias("max_population")
).display()

country,total_population
USA,null
INDIA,9999
IRAN,456789
INDIA,10000
USA,null
USA,500000000
USA,null
INDIA,1200000000
CHINA,1500000000


country,count(1),max_population
USA,4,500000000
INDIA,3,1200000000
IRAN,1,456789
CHINA,1,1500000000


##### HAVING function

In [0]:
%sql
select * from population;

country,total_population_2023,total_population_2022
CHINA,null,1400000000
USA,789456123,780000000
AFRICA,99999999,null
INDIA,331002651,1300000000
USA,500000000,450000000
USA,null,280000000
INDIA,1200000000,null
CHINA,null,1450000000


In [0]:
%sql
select country, count(*) as count, max(total_population_2023) from population group by country having count > 2;

country,count,max(total_population_2023)
USA,3,789456123


In [0]:
from pyspark.sql.functions import col

df1.display()
df1.groupby("country") \
    .agg(
        count("*").alias("count"),
        max(col("total_population")).alias("total")
    ) \
    .filter(
        col("count") > 2
    ).display()

country,total_population
USA,null
INDIA,9999
IRAN,456789
INDIA,10000
USA,null
USA,500000000
USA,300000000
INDIA,1200000000
CHINA,1500000000


country,count,total
USA,4,500000000
INDIA,3,1200000000
